# Muscle Epigenetic Clock
Building an age prediction model from GTEx skeletal muscle DNA methylation data.

**Method**: ElasticNet regression with Leave-One-Out cross-validation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import cross_val_predict, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Join Data

In [ ]:
# Load methylation data (CpGs as rows, samples as columns)
meth = pd.read_csv('GTEx_Muscle.meth.csv', sep='\t', index_col=0)
print(f"Methylation: {meth.shape[0]:,} CpGs × {meth.shape[1]} samples")

# Load annotation (variables as rows, samples as columns)
anno = pd.read_csv('GTEx_Muscle.anno.csv', sep='\t', index_col=0).T
print(f"Annotation: {anno.shape[0]} samples × {anno.shape[1]} variables")
anno[['age', 'Sex', 'tissue']].head(10)

In [ ]:
# Convert age ranges to midpoints
def age_midpoint(age_str):
    lo, hi = map(int, age_str.split('-'))
    return (lo + hi) / 2

anno['age_numeric'] = anno['age'].apply(age_midpoint)
print("Age distribution:")
print(f"  Mean: {anno['age_numeric'].mean():.1f} years")
print(f"  SD:   {anno['age_numeric'].std():.1f} years")
print(f"  Range: {anno['age_numeric'].min():.0f}-{anno['age_numeric'].max():.0f} years")

In [ ]:
# Prepare data matrices
X = meth.T.values.astype(np.float32)  # samples × CpGs
y = anno.loc[meth.columns, 'age_numeric'].values.astype(np.float32)
cpg_names = meth.index.values
print(f"X: {X.shape}, y: {y.shape}")

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Age distribution
axes[0].hist(y, bins=8, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Age (midpoint)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Age Distribution (n={len(y)})')
axes[0].axvline(y.mean(), color='red', linestyle='--', label=f'Mean={y.mean():.1f}')
axes[0].legend()

# Sex distribution
sex_counts = anno['Sex'].value_counts()
axes[1].bar(['Male (1)', 'Female (2)'], [sex_counts.get('1', 0), sex_counts.get('2', 0)], 
            color=['steelblue', 'coral'], edgecolor='black')
axes[1].set_ylabel('Count')
axes[1].set_title('Sex Distribution')

# Global methylation
sample_vals = X[:, ::1000].flatten()
axes[2].hist(sample_vals, bins=50, edgecolor='none', alpha=0.7)
axes[2].set_xlabel('Beta value')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Global Methylation Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Compute CpG-age correlations (vectorized)
X = np.nan_to_num(X, nan=0.5, posinf=1.0, neginf=0.0)
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_std[X_std == 0] = 1
X_norm = (X - X_mean) / X_std
y_norm = (y - y.mean()) / y.std()
correlations = (X_norm.T @ y_norm) / len(y)

print(f"Correlation range: [{correlations.min():.4f}, {correlations.max():.4f}]")

# Top correlations
top_pos_idx = np.argsort(correlations)[-5:][::-1]
top_neg_idx = np.argsort(correlations)[:5]

print("\nTop 5 positively correlated CpGs:")
for i in top_pos_idx:
    print(f"  {cpg_names[i]}: r = {correlations[i]:.4f}")

print("\nTop 5 negatively correlated CpGs:")
for i in top_neg_idx:
    print(f"  {cpg_names[i]}: r = {correlations[i]:.4f}")

In [ ]:
# Visualize top CpGs
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
top_cpgs_idx = list(top_pos_idx[:3]) + list(top_neg_idx[:3])
for idx, ax in zip(top_cpgs_idx, axes.flat):
    ax.scatter(y, X[:, idx], alpha=0.7, s=60, edgecolor='black')
    ax.set_xlabel('Age')
    ax.set_ylabel('Beta')
    ax.set_title(f'{cpg_names[idx]}\nr = {correlations[idx]:.3f}')
    z = np.polyfit(y, X[:, idx], 1)
    ax.plot(sorted(y), np.polyval(z, sorted(y)), 'r--', lw=2)
plt.suptitle('Top Age-Correlated CpGs', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(correlations, bins=100, edgecolor='none', alpha=0.7)
ax.axvline(0.5, color='red', linestyle='--', lw=2, label='|r| = 0.5 threshold')
ax.axvline(-0.5, color='red', linestyle='--', lw=2)
ax.set_xlabel('Pearson r with Age', fontsize=12)
ax.set_ylabel('Number of CpGs', fontsize=12)
ax.set_title('CpG-Age Correlation Distribution', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Feature Selection & Clock Building

In [ ]:
# Stringent filter: |r| > 0.5
sig_mask = np.abs(correlations) > 0.5
n_sig = sig_mask.sum()
print(f"CpGs with |r| > 0.5: {n_sig:,} ({100*n_sig/len(correlations):.2f}%)")

X_filtered = X[:, sig_mask]
cpg_filtered = cpg_names[sig_mask]
corr_filtered = correlations[sig_mask]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)
print(f"Filtered data: {X_scaled.shape}")

In [ ]:
# ElasticNet with strong regularization
model = ElasticNetCV(
    l1_ratio=[0.5, 0.7, 0.9, 0.95, 0.99, 1.0],
    alphas=np.logspace(-2, 2, 100),
    cv=5,
    max_iter=20000,
    random_state=42,
    n_jobs=-1
)

# Leave-One-Out CV
print("Running LOO-CV (47 folds)...")
loo = LeaveOneOut()
y_pred_loo = cross_val_predict(model, X_scaled, y, cv=loo, n_jobs=-1)
model.fit(X_scaled, y)

print(f"Best alpha: {model.alpha_:.4f}")
print(f"Best l1_ratio: {model.l1_ratio_:.2f}")
n_coefs = int((model.coef_ != 0).sum())
print(f"Non-zero coefficients: {n_coefs} / {len(model.coef_)}")

## 4. Model Evaluation

In [ ]:
# Calculate metrics
r, p = pearsonr(y, y_pred_loo)
mae = np.mean(np.abs(y - y_pred_loo))
rmse = np.sqrt(np.mean((y - y_pred_loo)**2))
r2 = 1 - np.sum((y - y_pred_loo)**2) / np.sum((y - y.mean())**2)

print("="*50)
print("MUSCLE CLOCK PERFORMANCE (Leave-One-Out CV)")
print("="*50)
print(f"Pearson r:     {r:.4f} (p = {p:.2e})")
print(f"R²:            {r2:.4f}")
print(f"MAE:           {mae:.2f} years")
print(f"RMSE:          {rmse:.2f} years")
print(f"Clock CpGs:    {n_coefs}")
print("="*50)

In [ ]:
# Predicted vs Actual plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(y, y_pred_loo, alpha=0.7, s=80, edgecolor='black', c='steelblue')
lims = [30, 80]
ax.plot(lims, lims, 'k--', lw=2, label='Perfect prediction')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Chronological Age (years)', fontsize=12)
ax.set_ylabel('Predicted Age (years)', fontsize=12)
ax.set_title(f'Muscle Clock: LOO-CV\nr = {r:.3f}, MAE = {mae:.1f} years', fontsize=14)
ax.legend()
ax.set_aspect('equal')

ax = axes[1]
residuals = y_pred_loo - y
ax.scatter(y, residuals, alpha=0.7, s=80, edgecolor='black', c='coral')
ax.axhline(0, color='black', linestyle='--', lw=2)
ax.set_xlabel('Chronological Age (years)', fontsize=12)
ax.set_ylabel('Residual (Predicted - Actual)', fontsize=12)
ax.set_title(f'Age Acceleration\nMean = {residuals.mean():.2f}, SD = {residuals.std():.2f}', fontsize=14)

plt.tight_layout()
plt.show()

## 5. Overfitting Diagnostic

In [ ]:
y_pred_train = model.predict(X_scaled)
r_train, _ = pearsonr(y, y_pred_train)
mae_train = np.mean(np.abs(y - y_pred_train))

print("OVERFITTING DIAGNOSTIC")
print("-" * 50)
print(f"{'Metric':<20} {'Training':<12} {'LOO-CV':<12} {'Gap':<10}")
print("-" * 50)
print(f"{'Pearson r':<20} {r_train:<12.4f} {r:<12.4f} {r_train-r:<10.4f}")
print(f"{'MAE (years)':<20} {mae_train:<12.2f} {mae:<12.2f} {mae-mae_train:<10.2f}")
print("-" * 50)

gap = r_train - r
if gap < 0.1:
    print("✓ GOOD: Minimal overfitting - model generalizes well")
elif gap < 0.2:
    print("⚠ ACCEPTABLE: Moderate overfitting for small n")
else:
    print("✗ HIGH: Consider more regularization")

## 6. Clock CpGs Analysis

In [ ]:
# Extract clock coefficients
coef_df = pd.DataFrame({
    'CpG': cpg_filtered,
    'Coefficient': model.coef_,
    'Correlation': corr_filtered
})
coef_df = coef_df[coef_df['Coefficient'] != 0].sort_values('Coefficient', key=abs, ascending=False)
print(f"Clock CpGs: {len(coef_df)}")
print(f"\nTop 20 by |coefficient|:")
coef_df.head(20)

In [ ]:
# Coefficient distribution
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(coef_df['Coefficient'], bins=30, edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', lw=2)
ax.set_xlabel('Coefficient Value', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title(f'Clock Coefficient Distribution (n={len(coef_df)} CpGs)', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Positive coefficients: {(coef_df['Coefficient'] > 0).sum()}")
print(f"Negative coefficients: {(coef_df['Coefficient'] < 0).sum()}")

In [ ]:
# Save clock
coef_df.to_csv('muscle_clock_coefficients.csv', index=False)
print(f"Saved clock to muscle_clock_coefficients.csv")

## 7. Summary

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Samples', 'Total CpGs', 'Pre-filtered (|r|>0.5)', 'Clock CpGs',
               'LOO-CV Pearson r', 'LOO-CV R²', 'LOO-CV MAE', 'LOO-CV RMSE',
               'Train-CV Gap (r)', 'Best Alpha', 'Best L1 Ratio'],
    'Value': [len(y), f"{len(cpg_names):,}", n_sig, n_coefs,
              f"{r:.4f}", f"{r2:.4f}", f"{mae:.2f} years", f"{rmse:.2f} years",
              f"{gap:.4f}", f"{model.alpha_:.4f}", f"{model.l1_ratio_:.2f}"]
})
print(summary.to_string(index=False))